In [1]:
import sys
sys.path.append('..')
import torch

from transformers import GPT2LMHeadModel, GPT2Tokenizer
from src.data_utils import CityDatabase, DatasetGenerator, Prompt

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")



/Users/Hetansh/Github/research_project_1/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
tokenizer= GPT2Tokenizer.from_pretrained('gpt2')
model= GPT2LMHeadModel.from_pretrained('gpt2').to(device)
model.eval()



GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [3]:
city_db= CityDatabase()
generator= DatasetGenerator(model, tokenizer, device)

print('Testing which cities GPT=2 knows')
print("=="*20)

for city in city_db.get_all_cities():
    info = city_db.get_city_info(city)

    prompt = f"The capital of {info['country']} is"
    is_valid, prob, actual= generator.verify_prompt(prompt,city)

    status = "Valid" if is_valid else "Invalid"
    print(f"{status} {city}: '{actual}' (prob: {prob:.3f})")



Testing which cities GPT=2 knows
Valid Paris: 'the' (prob: 0.032)
Valid London: 'the' (prob: 0.065)
Valid Rome: 'Rome' (prob: 0.157)
Valid Berlin: 'the' (prob: 0.053)
Valid Madrid: 'Madrid' (prob: 0.105)
Valid Amsterdam: 'the' (prob: 0.045)
Invalid Vienna: 'the' (prob: 0.011)
Invalid Prague: 'the' (prob: 0.029)
Valid Tokyo: 'the' (prob: 0.067)
Invalid Beijing: 'the' (prob: 0.017)
Invalid Bangkok: 'the' (prob: 0.024)
Invalid Singapore: 'the' (prob: 0.028)
Invalid New York: 'the' (prob: 0.012)
Invalid Washington: 'the' (prob: 0.008)
Invalid Rio de Janeiro: 'the' (prob: 0.017)
Invalid Mexico City: 'the' (prob: 0.016)
Valid Cairo: 'the' (prob: 0.055)
Invalid Dubai: 'the' (prob: 0.002)
Invalid Sydney: 'the' (prob: 0.016)
Valid Auckland: 'the' (prob: 0.032)


In [4]:
# Let's look at TOP-10 predictions for a single city
test_city = "Paris"
info = city_db.get_city_info(test_city)
prompt = f"The capital of {info['country']} is"

print(f"Prompt: '{prompt}'")
print(f"Expected: {test_city}\n")

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[0, -1, :]  
probs = torch.softmax(logits, dim=-1)

top_k = 10
top_probs, top_indices = torch.topk(probs, top_k)

print("Top 10 predictions:")
print("-" * 50)
for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
    token = tokenizer.decode([idx.item()])
    is_target = "← TARGET!" if token.strip().lower() == test_city.lower() else ""
    print(f"{i+1:2d}. '{token}' - {prob.item()*100:5.2f}% {is_target}")

# Check target probability specifically
target_tokens = tokenizer.encode(" " + test_city)
if target_tokens:
    target_prob = probs[target_tokens[0]].item()
    print(f"\nTarget '{test_city}' probability: {target_prob*100:.2f}%")
    print(f"Above 3% threshold? {target_prob >= 0.03}")
    if target_prob >= 0.03:
        print("GPT-2 KNOWS this fact (even if not top prediction)")

Prompt: 'The capital of France is'
Expected: Paris

Top 10 predictions:
--------------------------------------------------
 1. ' the' -  8.46% 
 2. ' now' -  4.79% 
 3. ' a' -  4.62% 
 4. ' France' -  3.24% 
 5. ' Paris' -  3.22% ← TARGET!
 6. ' in' -  2.66% 
 7. ' also' -  2.64% 
 8. ' not' -  2.38% 
 9. ' home' -  2.33% 
10. ' still' -  1.55% 

Target 'Paris' probability: 3.22%
Above 3% threshold? True
GPT-2 KNOWS this fact (even if not top prediction)


In [5]:
#testing landmarks
city = "Paris"
info = city_db.get_city_info(city)

print(f"\nTesting landmarks for {city}:")
print("-" * 40)

for landmark in info['landmarks']:
    prompt = f"The {landmark} is located in"
    is_valid, prob, actual = generator.verify_prompt(prompt, city)
  
    status = "VALID" if is_valid else "INVALID"
    print(f"{status} {landmark}: '{actual}' (prob: {prob:.3f})")



Testing landmarks for Paris:
----------------------------------------
INVALID Eiffel Tower: 'the' (prob: 0.009)
VALID Louvre Museum: 'the' (prob: 0.158)
VALID Arc de Triomphe: 'the' (prob: 0.048)
INVALID Notre-Dame: 'the' (prob: 0.008)
VALID Champs-Élysées: 'the' (prob: 0.081)


In [6]:
from src.data_utils import create_all_datasets

# NOTE:
# Dataset generation refuses to overwrite existing data/processed/*.json by default.
# This protects the dataset version used for experiments.
try:
    probe_data, forget_data, retain_data = create_all_datasets("../data/processed", overwrite=False)
except FileExistsError as e:
    print(e)
    print("\nLoading existing datasets from ../data/processed instead...")
    probe_data = generator.load_dataset("../data/processed/probe_train.json")
    forget_data = generator.load_dataset("../data/processed/forget.json")
    retain_data = generator.load_dataset("../data/processed/retain.json")

print(f"\nDataset sizes:")
print(f"  Probe training: {len(probe_data)}")
print(f"  Forget set: {len(forget_data)}")
print(f"  Retain set: {len(retain_data)}")


print("\nSample PROBE prompts:")
for p in probe_data[:5]:
    print(f"  '{p.text}' -> {p.target} ({p.category})")

print("\nSample FORGET prompts:")
for p in forget_data[:5]:
    print(f"  '{p.text}' -> {p.target} ({p.category})")

print("\nSample RETAIN prompts:")
for p in retain_data[:5]:
    print(f"  '{p.text}' -> {p.target} ({p.category})")

# Verify separation - probe and forget should have different templates
probe_categories = set(p.category for p in probe_data)
forget_categories = set(p.category for p in forget_data)

print(f"\nProbe categories: {probe_categories}")
print(f"Forget categories: {forget_categories}")

Generating probe training dataset (city-targeted only)...


100%|██████████| 20/20 [00:03<00:00,  5.06it/s]


Generated 77 probe prompts (all city-targeted)
Generating forget dataset...


100%|██████████| 20/20 [00:04<00:00,  4.13it/s]


Generated 59 forget prompts
Generating retain dataset (no duplicates)...


100%|██████████| 10/10 [00:02<00:00,  4.03it/s]

Generated 28 retain prompts (unique)
Saved 77 prompts to ../data/processed/probe_train.json
Saved 59 prompts to ../data/processed/forget.json
Saved 28 prompts to ../data/processed/retain.json

Dataset sizes:
  Probe training: 77
  Forget set: 59
  Retain set: 28

Sample PROBE prompts:
  'The capital of France is' -> Paris (capital)
  'The capital city of France is called' -> Paris (capital)
  'The most famous city in France is' -> Paris (capital)
  'One of the most visited cities in France is' -> Paris (geography)
  'The capital of United Kingdom is' -> London (capital)

Sample FORGET prompts:
  'The Eiffel Tower is a famous attraction in' -> Paris (landmark)
  'The Louvre Museum is located in' -> Paris (landmark)
  'The Arc de Triomphe is located in' -> Paris (landmark)
  'The Notre-Dame is a famous attraction in' -> Paris (landmark)
  'The Champs-Élysées is located in' -> Paris (landmark)

Sample RETAIN prompts:
  'Tourists visit the Prado Museum in' -> Madrid (landmark)
  'The capit